# Анализ качества работы нейронной сети для распознавания моделей одежды в Keras

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import utils
from tensorflow.keras.preprocessing import image
from google.colab import files
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

## Подготовка данных для обучения сети

**Загружаем набор данных**

In [ ]:
# В Keras встроены средства работы с популярными наборами данных
# (x_train, y_train) - набор данных для обучения
# (x_test, y_test) - набор данных для тестирования
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

Список с названиями классов

In [ ]:
classes = ['футболка', 'брюки', 'свитер', 'платье', 'пальто', 'туфли', 'рубашка', 'кроссовки', 'сумка', 'ботинки']

Просматриваем примеры изображений

In [ ]:
plt.figure(figsize=(10,10))
for i in range(100,150):
    plt.subplot(5,10,i-100+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    plt.xlabel(classes[y_train[i]])

**Преобразование размерности данных в наборе**

In [ ]:
x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)

**Нормализация данных**

In [ ]:
# Векторизованные операции
# Применяются к каждому элементу массива отдельно
x_train = x_train / 255
x_test = x_test / 255

**Работа с правильными ответами**

In [ ]:
n = 0

In [ ]:
print(y_train[n])

**Преобразуем метки в формат one hot encoding**

In [ ]:
y_train = utils.to_categorical(y_train, 10)

In [ ]:
y_test = utils.to_categorical(y_test, 10)

**Правильный ответ в формате one hot encoding**

In [ ]:
print(y_train[n])

## Создаем нейронную сеть

**Создаем последовательную модель**

In [ ]:
# Создаем последовательную модель
model = Sequential()
# Входной полносвязный слой, 800 нейронов, 784 входа в каждый нейрон
model.add(Dense(800, input_dim=784, activation="relu"))
# Выходной полносвязный слой, 10 нейронов (по количеству выходных классов)
model.add(Dense(10, activation="softmax"))

**Компилируем сеть**

In [ ]:
model.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])

print(model.summary())

## Обучаем нейронную сеть

In [ ]:
history = model.fit(x_train, y_train,
                    batch_size=200,
                    epochs=100,
                    validation_split=0.2,
                    verbose=1)

## Сохраняем нейронную сеть для последующего использования

In [ ]:
model.save('fashion_mnist_dense.h5')

## Оценка качества обучения

Проверка качества работы на наборе данных для тестирования

In [ ]:
scores = model.evaluate(x_test, y_test, verbose=1)

In [ ]:
print("Доля верных ответов на тестовых данных, в процентах:", round(scores[1] * 100, 4))

## Используем сеть для распознавания предметов одежды

In [ ]:
n_rec = 496

In [ ]:
plt.imshow(x_test[n_rec].reshape(28, 28), cmap=plt.cm.binary)
plt.show()

**Меняем размерность изображения и нормализуем его**

In [ ]:
x = x_test[n_rec]
x = np.expand_dims(x, axis=0)

**Запускаем распознавание**

In [ ]:
prediction = model.predict(x)

**Печатаем результаты распознавания**

In [ ]:
prediction

**Печатаем правильный ответ**

In [ ]:
label = np.argmax(y_test[n_rec])
print("Номер класса:", label)
print("Название класса:", classes[label])

## Загружаем свою картинку

In [ ]:
files.upload()

Проверяем загрузку картинки

In [ ]:
!ls

Загружаем картинку из файла

In [ ]:
img_path = 'tshirt.jpg'
img = image.load_img(img_path, target_size=(28, 28), color_mode = "grayscale")

Показываем картинку

In [ ]:
plt.imshow(img.convert('RGBA'))
plt.show()

Преобразуем картинку для обработки нейронной сетью

In [ ]:
# Преобразуем картинку в массив
x = image.img_to_array(img)
# Меняем форму массива в плоский вектор
x = x.reshape(1, 784)
# Инвертируем изображение
x = 255 - x
# Нормализуем изображение
x /= 255

Запускаем распознавание

In [ ]:
prediction = model.predict(x)

Результаты распознавания

In [ ]:
prediction

In [ ]:
prediction = np.argmax(prediction)
print("Номер класса:", prediction)
print("Название класса:", classes[prediction])

## Keras Tuner

Keras Tuner - это простая среда для автоматизации настройки оптимальных гиперпараметров. Поставляется со встроенными алгоритмами случайного поиска, гиперполосной оптимизации и байесовской оптимизации.<br/>

Последовательность настройки гиперпараметров:
*   Определить модель
*   Указать гиперпараметры для настройки
*   Определить пространство поиска
*   Определить алгоритм поиска

### Определение модели
Модель, которая создается для гипертюнинга, называется *гипермоделью*.
Есть два способа построить гипермодель:
*  Используя функцию построителя модели
*  Используя подкласс [HyperModel](https://keras.io/guides/keras_tuner/getting_started/#you-can-use-a-hypermodel-subclass-instead-of-a-modelbuilding-function) API Keras Tuner
Воспользуемся первым способом. Функция построения модели использует определенные гиперпараметры для возврата скомпилированной модели.

### Гиперпараметры для настройки
Гмпермодель очень похожа на неглубокую DNN, которая была обучена ранее, за исключением того, что будут настроены четыре гиперпараметра модели:
1.   Количество скрытых слоев
2.   Количество нейронов в каждом скрытом слое
3.   Процент выпадения после каждого скрытого слоя
4.   Скорость обучения оптимизатора Adam
    
    
    

### Определение пространства поиска
Это делается путем передачи объекта HyperParameters в качестве параметра функции построения модели, которая настраивает гиперпараметры. Используем следующие параметры:
*   hp.Int() для определения области поиска по количеству скрытых слоев и количества нейронов в каждом скрытом слое. Позволяет определять минимальные и максимальные значения, а также размер шага для увеличения.
*   hp.Float() для определения области поиска параметра droput. Похоже на hp.In(), за исключением того, что принимает плавающие значения.
*   hp.Choice() для определения области поиска скорости обучения. Позволяет определять дискретные значения.
Более подробную информацию обо всех доступных методах и их использовании можно найти в [официальной документации](https://keras.io/api/keras_tuner/hyperparameters/).




In [ ]:
!pip install keras-tuner -q

In [ ]:
import kerastuner as kt

In [ ]:
def build_model(hp):
    """
    Builds model and sets up hyperparameter space to search.

    Parameters
    ----------
    hp : HyperParameter object
        Configures hyperparameters to tune.

    Returns
    -------
    model : keras model
        Compiled model with hyperparameters to tune.
    """
    # Initialize sequential API and start building model.
    model = keras.Sequential()
    # model.add(keras.layers.Flatten(input_shape=(28,28)))

    # Tune the number of hidden layers and units in each.
    # Number of hidden layers: 1 - 5
    # Number of Units: 32 - 512 with stepsize of 32
    for i in range(1, hp.Int("num_layers", 1, 5)):
        model.add(
            keras.layers.Dense(
                units=hp.Int("units_" + str(i), min_value=32, max_value=512, step=32),
                activation="relu", input_dim=784,)
            )

        # Tune dropout layer with values from 0 - 0.3 with stepsize of 0.1.
        model.add(keras.layers.Dropout(hp.Float("dropout_" + str(i), 0, 0.3, step=0.1)))

    # Add output layer.
    model.add(keras.layers.Dense(units=10, activation="softmax"))

    # Tune learning rate for Adam optimizer with values from 0.01, 0.001, or 0.0001
    hp_learning_rate = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])

    # Define optimizer, loss, and metrics
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss="categorical_crossentropy",
                  # loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])

    return model

# model.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])

### Определение алгоритма поиска
Для поиска оптимальных гиперпараметров будем использовать алгоритм Hyperband. Это новый подход, разработанный специально для оптимизации гиперпараметров. Подробное описание алгоритма можно найти [здесь](https://jmlr.org/papers/v18/16-558.html).
Основная идея алгоритма: Hyperband использует сетку в стиле спортивного чемпионата и начинает со случайного выбора большого количества моделей со случайными перестановками гиперпараметров из области поиска. Каждая модель обучается в течение нескольких эпох, затем наиболее эффективная половина моделей переходит в следующий раунд.<br/>
Чтобы создать экземпляр тюнера, нужно определить следующие гиперпараметры:
*   Гипермодель (функция построителя моделей).
*   Цель - направление поиска (минимальное или максимальное), автоматически определяется для встроенных метрик, для пользовательских метрик можно использовать kerastuner.Objective.
*   Factor и max_epochs используются для расчета количества моделей путем взятия логарифмического базового фактора max_epochs + 1. Это число округляется до ближайшего целого числа.
*   Итерации Hyperband используются для управления ресурсами. Гиперполосные итерации - это количество повторений по всему алгоритму поиска.
*   Каталог сохраняет журналы и контрольные точки для каждого пробного запуска во время поиска по гиперпараметрам, что позволяет продолжить поиск с того места, где остановились в последний раз. Данную опцию можно отключить, установив дополнительный гиперпараметр «overwrite» = True.
*   Project_name используется для отличия от других запусков и является подкаталогом в каталоге.<br/>
Со списком всех доступных параметров можно ознакомиться в [документации](https://keras.io/api/keras_tuner/tuners/hyperband/).






In [ ]:
# Instantiate the tuner
tuner = kt.Hyperband(build_model,
                     objective="val_accuracy",
                     max_epochs=100,
                     factor=3,
                     hyperband_iterations=10,
                     directory="kt_dir",
                     project_name="kt_hyperband")

Сводка пространства поиска

In [ ]:
tuner.search_space_summary()

Установим обратные вызовы с ранней остановкой, чтобы прервать обучение, если показатели модели не улучшаются.

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20)

# Запускаем поиск
NUM_EPOCHS = 100
tuner.search(x_train, y_train, epochs=NUM_EPOCHS, validation_split=0.2, callbacks=[stop_early], verbose=1)

После завершения поиска можно получить лучшие гиперпараметры и переобучить модель.

In [ ]:
# Get the optimal hyperparameters from the results
best_hps=tuner.get_best_hyperparameters()[0]

# Build model
h_model = tuner.hypermodel.build(best_hps)

# Train the hypertuned model
h_model.fit(x_train, y_train, epochs=NUM_EPOCHS, validation_split=0.2, callbacks=[stop_early], verbose=1)

Оценить гипертонастроенную модель на тестовом наборе.